# xbubekir-KI trainieren (Qwen3 1,7B → eigene xbubekir-KI im Browser)

**So geht's:** Oben im Menü *Laufzeit → Laufzeittyp ändern → **T4 GPU*** wählen und speichern. Dann die Zellen **der Reihe nach** mit ▶ starten.
Jede Zelle sagt am Ende, ob alles geklappt hat. Wenn etwas rot wird: Screenshot an Claude schicken.

**Einmalig vorher:** Links auf das Schlüssel-Symbol (*Secrets*) tippen → *Neues Secret* → Name `HF_TOKEN`, Wert = dein Hugging-Face-Token (mit Schreibrecht) → Schalter *Notebook-Zugriff* einschalten.

Dauer insgesamt etwa 45–60 Minuten. Grundmodell: Qwen3-1.7B (Apache-2.0).

## 1. Pakete und Trainingsdaten

In [ ]:
!pip -q uninstall -y torchao  # in Colab vorinstalliert, aber zu alt für peft (wird nicht gebraucht)
!pip -q install "transformers>=4.51,<5" "peft>=0.14" "accelerate>=1.2" huggingface_hub
import json, urllib.request
BRANCH = "main"  # aus diesem GitHub-Zweig werden die Trainingsdaten geladen
BASE = f"https://raw.githubusercontent.com/xbubekir68/xbubekir68.github.io/{BRANCH}/xbubekir/training/"
data = {}
for f in ["train.jsonl", "eval.jsonl"]:
    data[f] = [json.loads(l) for l in urllib.request.urlopen(BASE + f).read().decode("utf-8").splitlines() if l.strip()]
print("✅ Trainingsgespräche:", len(data["train.jsonl"]), "| Prüfgespräche:", len(data["eval.jsonl"]))

## 2. Grundmodell laden

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
assert torch.cuda.is_available(), "Keine GPU! Laufzeit → Laufzeittyp ändern → T4 GPU"
MODEL = "Qwen/Qwen3-1.7B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32, device_map="cuda")
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
print("✅ Modell geladen:", MODEL)

## 3. Gespräche in Lernbeispiele umwandeln (gelernt wird nur, was xbubekir antwortet)

In [ ]:
MAXLEN = 1536
END = "<|im_end|>\n"
def examples(conv):
    msgs = conv["messages"]
    for j, m in enumerate(msgs):
        if m["role"] != "assistant":
            continue
        prompt = tok.apply_chat_template(msgs[:j], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        p = tok(prompt, add_special_tokens=False)["input_ids"]
        a = tok(m["content"] + END, add_special_tokens=False)["input_ids"]
        ids = (p + a)[-MAXLEN:]
        labels = ([-100] * len(p) + a)[-MAXLEN:]
        yield {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}
train = [e for c in data["train.jsonl"] for e in examples(c)]
evals = [e for c in data["eval.jsonl"] for e in examples(c)]
x = train[0]
print("Beispiel – Eingabe:\n", tok.decode(x["input_ids"])[-400:])
print("\nBeispiel – davon gelernt:\n", tok.decode([t for t, l in zip(x["input_ids"], x["labels"]) if l != -100]))
print("\n✅ Lernbeispiele:", len(train), "| Prüfbeispiele:", len(evals))

## 4. Nachtrainieren (LoRA) – dauert etwa 20–30 Minuten

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()
args = TrainingArguments(output_dir="/content/run", num_train_epochs=2, per_device_train_batch_size=2,
    gradient_accumulation_steps=8, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
    fp16=True, logging_steps=10, eval_strategy="epoch", save_strategy="no", report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=evals,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
trainer.train()
print("✅ Training fertig. Prüf-Verlust:", round(trainer.evaluate()["eval_loss"], 3))

## 5. Probelauf: 10 Testfragen – bitte die Antworten anschauen

In [ ]:
model.eval()
model.config.use_cache = True
PERSONA = json.loads(urllib.request.urlopen(BASE + "persona.json").read().decode("utf-8"))["system"]
UEBUNG = next(c["messages"][0]["content"] for c in data["train.jsonl"] if "Übungspartner" in c["messages"][0]["content"])
tests = [(PERSONA, "Wer bist du?"), (PERSONA, "Kannst du im Internet suchen?"), (PERSONA, "Erklär mir kurz, was ein Verb ist."),
         (PERSONA, "Schreib eine kurze Nachricht an meinen Freund, dass ich später komme."), (PERSONA, "Was ist 7 mal 8?"),
         (UEBUNG, "مَرْحَبًا"), (UEBUNG, "Frag mich ab."), (UEBUNG, "Fang du an."),
         (UEBUNG, "Gib mir einen Satz."), (UEBUNG, "weiß ich nicht")]
for system, q in tests:
    text = tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": q}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tok(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=160, do_sample=False)
    print("🧑", q, "\n🤖", tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip(), "\n" + "-" * 60)

## 6. Zusammenführen und speichern

In [ ]:
merged = model.merge_and_unload().half()
merged.save_pretrained("/content/xbubekir-hf", safe_serialization=True)
tok.save_pretrained("/content/xbubekir-hf")
del model, merged, trainer
import gc; gc.collect(); torch.cuda.empty_cache()
print("✅ Gespeichert in /content/xbubekir-hf")

## 7. Ins Browser-Format umwandeln (MLC, 4-Bit) – dauert etwa 5–10 Minuten

In [ ]:
!pip -q install --pre -U -f https://mlc.ai/wheels mlc-llm-nightly-cpu mlc-ai-nightly-cpu
!mlc_llm convert_weight /content/xbubekir-hf --quantization q4f16_1 --device cpu -o /content/xbubekir-mlc
# Konfiguration und Tokenizer vom offiziellen WebLLM-Modell übernehmen, damit es sicher zur fertigen Browser-Bibliothek passt
from huggingface_hub import hf_hub_download, list_repo_files
import shutil
OFFICIAL = "mlc-ai/Qwen3-1.7B-q4f16_1-MLC"
for f in list_repo_files(OFFICIAL):
    if f.endswith(".json") and not f.startswith(("ndarray-cache", "tensor-cache")) or f in ("merges.txt", "vocab.json"):
        shutil.copy(hf_hub_download(OFFICIAL, f), "/content/xbubekir-mlc/" + f)
import os; print("✅ Dateien:", sorted(os.listdir("/content/xbubekir-mlc")))

## 8. Auf Hugging Face hochladen

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi
api = HfApi(token=userdata.get("HF_TOKEN"))
user = api.whoami()["name"]
REPO = f"{user}/xbubekir-qwen3-1.7b-q4f16_1-MLC"
open("/content/xbubekir-mlc/README.md", "w").write(
    "---\nlicense: apache-2.0\nbase_model: Qwen/Qwen3-1.7B\n---\n# xbubekir (Qwen3-1.7B, nachtrainiert)\n\n"
    "Nachtrainierte Version von [Qwen/Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) (Apache-2.0) für die xbubekir-KI, "
    "im MLC-Format für WebLLM (q4f16_1).\n")
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path="/content/xbubekir-mlc", repo_id=REPO)
print("✅ Fertig! Schick Claude diese Adresse:\nhttps://huggingface.co/" + REPO)